# Name That LLM: two-stage LoRA classifier on Llama-3.1-8B, both GPUs, 5 folds (training)

Generated by `name_that_llm/kaggle_lora_train.py`; do not edit by hand.

* **Backbone:** first 24 of 32 layers of Llama-3.1-8B, LoRA r16 / alpha 32 on all linear layers, final norm off.
* **Head:** mean-pooled hidden states at 8 depths, a LayerNorm each, one linear layer.
* **Stage 1:** 1 epoch on up to 8,000 match-weighted Open Pool passages per model (draw seed 6), data-parallel on every GPU with the same global batch of 8. **Stage 2:** 2 epochs on train.csv.
* **Folds:** from the stage-1 weights, stage 2 runs once per prompt-grouped fold (5) leaving it out and scoring it, then once on all of train.csv. Outputs: `oof_logits.parquet` for every training passage, `fold0..4/`, `stage1/`, and the full fit at the top of the folder.

Needs **GPU T4 x2**, internet off, and three inputs: the competition data, `crtproduction/name-that-llm-open-pool` and `metaresearch/llama-3.1/Transformers/8b`.

To switch between the timing run and the full run, change `MODE` in the next cell.

In [ ]:
import glob
import json
import os
import time

import torch

MODE = 'timing'  # 'timing' or 'full'
T0 = time.time()
ON_KAGGLE = os.path.exists('/kaggle/input')
BASE_SPEC = {'model_type': 'llama', 'hidden_size': 4096, 'num_hidden_layers': 32}  # config.json fields of metaresearch/llama-3.1/Transformers/8b


def find_base(spec):
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f'attach the base model {spec}')


if ON_KAGGLE:
    attached = sorted(glob.glob('/kaggle/input/*/*'))
    print('attached inputs:', attached)
    comps = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'test.csv'))]
    assert comps, f'Attach the name-that-llm competition data; found {attached}'
    pools = glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True)
    assert pools, f'Attach crtproduction/name-that-llm-open-pool; found {attached}'
    assert torch.cuda.device_count() >= 2, 'pick the GPU T4 x2 accelerator for this notebook'
    paths = dict(comp=comps[0], pool=pools[0], base_dir=find_base(BASE_SPEC), out='lora_out_llama8b_full')
    SMOKE = False
else:  # smoke test from this project's notebooks/ folder, 2 CPU processes
    paths = dict(comp='../data/raw', pool='../data/external/open_pool/ntl_open_pool.parquet',
                 base_dir=os.environ.get('NTL_SMOKE_MODEL', '../data/interim/tiny_qwen3'),
                 out='../data/interim/lora_smoke_out_ddp')
    SMOKE = True

CFG = dict(
    keep=24, layers=[2, 4, 6, 8, 12, 16, 20, 24], r=16, alpha=32, dropout=0.05,
    max_len=384, bs=8,  # global batch, split across the GPUs
    stage1=dict(epochs=1, lr_lora=1e-4, lr_head=1e-3),
    stage2=dict(epochs=2, lr_lora=5e-5, lr_head=5e-4),
    n_per_label=8000, pool_seed=6, folds=5, seed=0,
    timing_steps=60, base=BASE_SPEC, mode=MODE, smoke=SMOKE, **paths,
)
if SMOKE:
    CFG.update(keep=2, layers=[1, 2], max_len=64, bs=4, n_per_label=10)
with open('cfg.json', 'w') as f:
    json.dump(CFG, f, indent=1)
print('mode', MODE, '| GPUs', torch.cuda.device_count(), '| base model', paths['base_dir'])

## Training code (copied from `name_that_llm/lora.py` and `lora_ddp.py`)

In [ ]:
%%writefile lora.py
"""Two-stage LoRA classifier on the first layers of a decoder LLM ("LoRA24" recipe).

Self-contained (torch + transformers only, no peft) so that kaggle_lora_train.py can copy it
verbatim into a Kaggle notebook. See references/lora-plan.md for the recipe and why.

* The backbone keeps only its first ``keep`` decoder layers; the final norm is switched off.
* Every linear layer inside them gets a LoRA adapter (fp32 parameters on frozen fp16 weights).
* A passage's features are its mean hidden state (first token left out) at several depths, each
  through its own LayerNorm, concatenated, then one linear layer to the classes.
* Training runs in fp16 autocast with loss scaling on T4s (no bf16 there); batches whose loss is
  not finite are skipped and counted.
* On TPU (PyTorch/XLA, see lora_ddp.py) the trunk is stored in bf16 and runs without autocast,
  and every batch is padded to a fixed set of shapes (``static=True``), because XLA compiles a new
  program for every new tensor shape.
"""

import contextlib
import json
import math
import os
import time

import numpy as np
import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update ``B @ A * alpha / r``."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, device=dev))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        delta = self.drop(x).to(self.lora_A.dtype) @ self.lora_A.T @ self.lora_B.T
        return self.base(x) + (delta * self.scale).to(x.dtype)


def add_lora(model: nn.Module, r: int, alpha: float, dropout: float) -> int:
    """Wrap every target linear layer of ``model`` in LoRALinear; returns how many."""
    targets = [
        (parent, name)
        for parent in model.modules()
        for name, child in parent.named_children()
        if isinstance(child, nn.Linear) and name in LORA_TARGETS
    ]
    for parent, name in targets:
        setattr(parent, name, LoRALinear(getattr(parent, name), r, alpha, dropout))
    return len(targets)


def load_trunk(base_dir: str, keep: int, dtype, device, attn_implementation=None) -> nn.Module:
    """The first ``keep`` decoder layers of the base model, frozen, final norm switched off."""
    cfg = AutoConfig.from_pretrained(base_dir)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, "layer_types", None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    kw = {"attn_implementation": attn_implementation} if attn_implementation else {}
    trunk = AutoModel.from_pretrained(base_dir, config=cfg, dtype=dtype, **kw)
    # .to(dtype) too: transformers before 4.56 ignores ``dtype=`` and would load fp32
    trunk = trunk.to(device=device, dtype=dtype)
    trunk.norm = nn.Identity()
    trunk.requires_grad_(False)
    return trunk


def no_autocast(device_type: str):
    """Autocast switched off (the head runs in fp32); XLA runs without autocast anyway."""
    if device_type in ("cuda", "cpu"):
        return torch.autocast(device_type=device_type, enabled=False)
    return contextlib.nullcontext()


class LoRAClassifier(nn.Module):
    def __init__(
        self, trunk: nn.Module, layers, n_classes: int, r=16, alpha=32, dropout=0.05, ckpt=True
    ):
        super().__init__()
        self.trunk, self.layers = trunk, list(layers)
        self.n_lora = add_lora(trunk, r, alpha, dropout)
        if ckpt:  # recompute activations in backward: far less memory for ~30% more time
            trunk.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )
        H = trunk.config.hidden_size
        dev = trunk.embed_tokens.weight.device
        self.norms = nn.ModuleList([nn.LayerNorm(H) for _ in self.layers]).to(dev)
        self.head = nn.Linear(H * len(self.layers), n_classes).to(dev)

    def forward(self, input_ids, attention_mask):
        hs = self.trunk(
            input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True
        ).hidden_states
        m = attention_mask.unsqueeze(-1).float()
        m[:, 0] = 0  # the first token is an attention sink; leave it out of the mean
        den = m.sum(1).clamp(min=1)
        with no_autocast(input_ids.device.type):
            z = torch.cat(
                [nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(self.layers, self.norms)],
                -1,
            )
            return self.head(z)

    def trainable_state(self) -> dict:
        """Only what training changes: LoRA matrices, LayerNorms and the head."""
        return {k: v.detach().cpu() for k, v in self.named_parameters() if v.requires_grad}


def tokenize(tok, texts, max_len: int) -> list[list[int]]:
    """Token ids of each passage, ending in EOS, and starting with BOS if the tokenizer has one
    (Llama does; Qwen has none, so Qwen ids are just the text and EOS). At most ``max_len``."""
    bos = [] if tok.bos_token_id is None else [tok.bos_token_id]
    ids = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=max_len - 1 - len(bos),
    )
    return [bos + x + [tok.eos_token_id] for x in ids["input_ids"]]


def batches(ids, bs: int, shuffle: bool, rng=None):
    """Index batches of similar length (less padding); batch order shuffled when training."""
    if not shuffle:
        order = np.argsort([len(x) for x in ids], kind="stable")
    else:  # sort within random chunks of 50 batches so batches still differ between epochs
        order = rng.permutation(len(ids))
        chunk = bs * 50
        order = np.concatenate(
            [
                sorted(order[i : i + chunk], key=lambda j: len(ids[j]))
                for i in range(0, len(order), chunk)
            ]
        )
    out = [order[i : i + bs] for i in range(0, len(order), bs)]
    if shuffle:
        out = [out[i] for i in rng.permutation(len(out))]
    return out


PAD_MULTIPLE = 64  # static shapes: sequence lengths are rounded up to a multiple of this


def static_len(ids, idx, max_len: int) -> int:
    """The padded length for a batch: its longest passage rounded up to ``PAD_MULTIPLE``, so XLA
    sees at most ``max_len / PAD_MULTIPLE`` sequence lengths."""
    longest = max(len(ids[i]) for i in idx)
    return min(max_len, -(-longest // PAD_MULTIPLE) * PAD_MULTIPLE)


def collate(tok, ids, idx, device, length: int | None = None):
    """Pad a batch to its longest passage, or to ``length`` (static shapes for XLA)."""
    feats = {"input_ids": [ids[i] for i in idx]}
    if length is None:
        enc = tok.pad(feats, return_tensors="pt")
    else:
        enc = tok.pad(feats, padding="max_length", max_length=length, return_tensors="pt")
    return enc["input_ids"].to(device), enc["attention_mask"].to(device)


def train_stage(
    model,
    tok,
    texts,
    y,
    *,
    epochs: float,
    lr_lora: float,
    lr_head: float,
    bs: int,
    max_len: int,
    n_classes: int,
    amp_dtype,
    seed: int = 0,
    max_steps: int | None = None,
    log_every: int = 50,
    log=print,
):
    """One training stage; class-weighted CE, AdamW, 10% warm-up then cosine decay."""
    device = model.head.weight.device
    rng = np.random.default_rng(seed)
    ids = tokenize(tok, texts, max_len)
    y = torch.tensor(np.array(y), device=device)
    counts = torch.bincount(y, minlength=n_classes).float()
    class_w = (counts.sum() / (n_classes * counts.clamp(min=1))).to(device)

    per_epoch = math.ceil(len(ids) / bs)
    total = math.ceil(epochs * per_epoch)
    if max_steps is not None:
        total = min(total, max_steps)
    lora = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
    opt = torch.optim.AdamW(
        [{"params": lora, "lr": lr_lora}, {"params": rest, "lr": lr_head}], weight_decay=0.01
    )
    warm = max(1, int(0.1 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (
            (s + 1) / warm
            if s < warm
            else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
        ),
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

    model.train()
    step, skipped, run_loss, run_n, t0 = 0, 0, 0.0, 0, time.time()
    while step < total:
        for idx in batches(ids, bs, shuffle=True, rng=rng):
            if step >= total:
                break
            input_ids, mask = collate(tok, ids, idx, device)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(input_ids, mask)
            loss = nn.functional.cross_entropy(logits.float(), y[idx], weight=class_w)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1  # fp16 overflow in the backbone; drop this batch
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(lora + rest, 1.0)
                scaler.step(opt)
                scaler.update()
                run_loss += loss.item()
                run_n += 1
            sched.step()
            step += 1
            if step % log_every == 0 or step == total:
                el = time.time() - t0
                log(
                    f"step {step}/{total} loss {run_loss / max(1, run_n):.4f} "
                    f"{el / step:.2f}s/step, ~{el / step * (total - step) / 60:.0f} min left, "
                    f"skipped {skipped}"
                )
                run_loss, run_n = 0.0, 0
    model.eval()
    return {"steps": total, "skipped": skipped, "seconds": time.time() - t0}


@torch.no_grad()
def predict_logits(
    model, tok, texts, *, bs: int, max_len: int, amp_dtype, static: bool = False
) -> np.ndarray:
    """Logits for ``texts``. ``static=True`` (XLA) pads every batch to ``bs`` rows and a length
    from ``static_len``; the padding rows are dropped again."""
    device = model.head.weight.device
    ids = tokenize(tok, texts, max_len)
    out = np.zeros((len(ids), model.head.out_features), np.float32)
    model.eval()
    for idx in batches(ids, bs, shuffle=False):
        rows = idx
        if static and len(idx) < bs:
            rows = np.concatenate([idx, np.repeat(idx[-1:], bs - len(idx))])
        length = static_len(ids, rows, max_len) if static else None
        input_ids, mask = collate(tok, ids, rows, device, length)
        with torch.autocast(
            device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"
        ):
            out[idx] = model(input_ids, mask).float().cpu().numpy()[: len(idx)]
    return out


def load_tokenizer(base_dir: str):
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def save_checkpoint(model, out_dir: str, meta: dict):
    from safetensors.torch import save_file

    os.makedirs(out_dir, exist_ok=True)
    save_file(
        {k: v.contiguous() for k, v in model.trainable_state().items()},
        os.path.join(out_dir, "lora_classifier.safetensors"),
    )
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)


def load_checkpoint(model, ckpt_dir: str):
    from safetensors.torch import load_file

    state = load_file(os.path.join(ckpt_dir, "lora_classifier.safetensors"))
    missing = model.load_state_dict(state, strict=False)
    assert not missing.unexpected_keys, missing.unexpected_keys
    return model


In [ ]:
%%writefile lora_ddp.py
"""Two-stage LoRA training on several GPUs or TPU chips, with stage 2 on every prompt-grouped fold.

Run with ``torchrun --standalone --nproc_per_node=N lora_ddp.py cfg.json`` (references/
lora-2gpu-5fold-plan.md). kaggle_lora_train.py copies this file and ``lora.py`` verbatim into the
notebook's working folder, so it imports ``lora`` as a sibling module there.

* Stage 1: 1 epoch on the Open Pool sample, data-parallel (DDP): every GPU builds the same batch
  list from the same seed and takes its share of each batch, so the global batch and the recipe
  are the same as on one GPU.
* Stage 2: from the stage-1 weights, once per fold leaving that fold out (its held-out logits make
  ``oof_logits.parquet``), then once on all of train.csv.
* A batch whose fp16 loss is not finite on any GPU is skipped on all of them, so no GPU waits
  forever for another one's gradients.
* Without a GPU it runs on CPU with the gloo backend (the local smoke test).

With ``"accelerator": "tpu"`` in cfg.json, run it once as ``python lora_ddp.py cfg.json``: it starts
one PyTorch/XLA process per TPU chip itself (``xmp.spawn``). The recipe is the same; what changes:

* The trunk is bf16 and runs without autocast or loss scaling (bf16 does not overflow like fp16),
  and gradient checkpointing is off (a TPU v5e chip has 16 GB, enough for the 8B trunk at batch 1).
* Static shapes: every rank pads to the same length, the longest passage of the whole global
  batch rounded up to 64 tokens, and a last batch smaller than ``bs`` is dropped. XLA then compiles
  one training program per length (at most 6 at max_len 384) instead of one per batch.
* Gradients are averaged with ``xm.reduce_gradients``, and the loss is read back to the host only
  when it is logged, since every read waits for the TPU.
* Every rank builds its model on the CPU from the same seed, a few ranks at a time so host RAM
  holds only a few copies of the trunk at once, then moves it to its chip.
"""

from datetime import timedelta
import gc
import json
import math
import os
import sys
import time

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
import torch
from torch import nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

try:  # flat files on Kaggle
    from lora import (
        LoRAClassifier,
        batches,
        collate,
        load_tokenizer,
        load_trunk,
        predict_logits,
        save_checkpoint,
        static_len,
        tokenize,
    )
except ImportError:  # inside this package
    from name_that_llm.lora import (
        LoRAClassifier,
        batches,
        collate,
        load_tokenizer,
        load_trunk,
        predict_logits,
        save_checkpoint,
        static_len,
        tokenize,
    )


def sample_pool(pool: pd.DataFrame, n_per_label: int, seed: int) -> pd.DataFrame:
    """Same draw as name_that_llm/dataset.py: weighted sampling without replacement by
    match_weight (Efraimidis-Spirakis), up to ``n_per_label`` passages per label."""
    parts = []
    for _, d in pool.groupby("model"):
        rng = np.random.default_rng([seed, len(parts)])
        keys = rng.random(len(d)) ** (1.0 / d.match_weight.clip(lower=1e-6).to_numpy())
        parts.append(d.iloc[np.argsort(-keys)[: min(n_per_label, len(d))]])
    return pd.concat(parts, ignore_index=True)


RANK = 0  # this process's rank, set in run()


def log0(*args):
    if RANK == 0:
        print(*args, flush=True)


class Comm:
    """What training needs from the process group: torch.distributed (NCCL on GPU, gloo on CPU)
    or PyTorch/XLA on TPU."""

    def __init__(self, xla: bool):
        self.xla = xla
        if xla:
            import torch_xla
            import torch_xla.core.xla_model as xm
            import torch_xla.runtime as xr

            self.xm = xm
            self.rank, self.world = xr.global_ordinal(), xr.world_size()
            self.device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
            self.sync = getattr(torch_xla, "sync", None) or xm.mark_step
            self.amp = torch.bfloat16  # the trunk's dtype; no autocast on XLA
            return
        use_cuda = torch.cuda.is_available()
        # a GPU waits at most 30 min for another one (rank 0 alone scores each held-out fold)
        dist.init_process_group("nccl" if use_cuda else "gloo", timeout=timedelta(minutes=30))
        self.rank, self.world = dist.get_rank(), dist.get_world_size()
        if use_cuda:
            torch.cuda.set_device(int(os.environ.get("LOCAL_RANK", self.rank)))
            self.device = torch.device("cuda", torch.cuda.current_device())
            major = torch.cuda.get_device_capability()[0]
            self.amp = torch.bfloat16 if major >= 8 else torch.float16
        else:
            self.device, self.amp = torch.device("cpu"), torch.float32

    def barrier(self, tag: str):
        if self.xla:
            self.xm.rendezvous(tag)
        else:
            dist.barrier()

    def close(self):
        if not self.xla:
            dist.destroy_process_group()


def train_stage_ddp(
    comm,
    fwd,
    model,
    tok,
    texts,
    y,
    *,
    epochs: float,
    lr_lora: float,
    lr_head: float,
    bs: int,
    max_len: int,
    n_classes: int,
    amp_dtype,
    seed: int = 0,
    max_steps: int | None = None,
    log_every: int = 50,
):
    """``lora.train_stage`` on several processes: same schedule, global batch ``bs``.

    ``fwd`` is the DDP wrapper on GPU/CPU (it averages the gradients in backward) and the bare
    model on XLA (gradients averaged by ``xm.reduce_gradients``)."""
    rank, world, xla = comm.rank, comm.world, comm.xla
    device = model.head.weight.device
    rng = np.random.default_rng(seed)
    ids = tokenize(tok, texts, max_len)
    y = np.asarray(y)
    counts = np.bincount(y, minlength=n_classes).astype(np.float32)
    class_w = torch.tensor(counts.sum() / (n_classes * np.maximum(counts, 1)), device=device)

    per_epoch = math.ceil(len(ids) / bs)
    total = math.ceil(epochs * per_epoch)
    if max_steps is not None:
        total = min(total, max_steps)
    lora = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
    opt = torch.optim.AdamW(
        [{"params": lora, "lr": lr_lora}, {"params": rest, "lr": lr_head}], weight_decay=0.01
    )
    warm = max(1, int(0.1 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (
            (s + 1) / warm
            if s < warm
            else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
        ),
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)
    # every rank needs a share of each batch; on XLA every batch also needs the same shape
    min_batch = bs if xla else world
    # the steps after the first few (and, on XLA, after most compiling) give the steady s/step
    warm_steps = min(20, total // 2)

    fwd.train()
    step, skipped, run_n, t0, t_warm = 0, 0, 0, time.time(), None
    run_loss = torch.zeros((), device=device) if xla else 0.0
    while step < total:
        for idx in batches(ids, bs, shuffle=True, rng=rng):  # same list on every rank
            if step >= total:
                break
            if len(idx) < min_batch:  # the last batch of an epoch; the same on every rank
                continue
            mine = idx[rank::world]
            length = static_len(ids, idx, max_len) if xla else None
            input_ids, mask = collate(tok, ids, mine, device, length)
            target = torch.as_tensor(y[mine]).to(device)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = fwd(input_ids, mask)
            loss = nn.functional.cross_entropy(logits.float(), target, weight=class_w)
            opt.zero_grad(set_to_none=True)
            if xla:
                loss.backward()
                comm.xm.reduce_gradients(opt)
                torch.nn.utils.clip_grad_norm_(lora + rest, 1.0)
                opt.step()
                comm.sync()  # run this step's graph on the TPU
                run_loss = run_loss + loss.detach()
                run_n += 1
            else:
                ok = torch.tensor(float(torch.isfinite(loss)), device=device)
                dist.all_reduce(ok, op=dist.ReduceOp.MIN)
                if ok.item() < 1:
                    skipped += 1  # fp16 overflow on some GPU; every GPU drops this batch
                else:
                    scaler.scale(loss).backward()  # DDP averages the gradients here
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(lora + rest, 1.0)
                    scaler.step(opt)
                    scaler.update()
                    run_loss += loss.item()
                    run_n += 1
            sched.step()
            step += 1
            if step == warm_steps:
                t_warm = time.time()
            if step % log_every == 0 or step == total:
                el = time.time() - t0
                mean = float(run_loss) / max(1, run_n)  # on XLA this waits for the TPU
                extra = ""
                if xla:
                    import torch_xla.debug.metrics as met

                    compiles = met.metric_data("CompileTime")
                    extra = f", {compiles[0] if compiles else 0} XLA compilations"
                log0(
                    f"step {step}/{total} loss {mean:.4f} "
                    f"{el / step:.2f}s/step, ~{el / step * (total - step) / 60:.0f} min left, "
                    f"skipped {skipped}{extra}"
                )
                run_loss = torch.zeros((), device=device) if xla else 0.0
                run_n = 0
    fwd.eval()
    t_end = time.time()
    out = {"steps": total, "skipped": skipped, "seconds": t_end - t0}
    if t_warm is not None and total > warm_steps:
        out["steady_s_per_step"] = (t_end - t_warm) / (total - warm_steps)
    return out


def build_model(comm, C: dict, n_classes: int):
    """The classifier on this rank's device with the same initial weights on every rank, and
    what to call for the forward pass (DDP on GPU/CPU, the bare model on XLA)."""
    if not comm.xla:
        torch.manual_seed(C["seed"])  # same LoRA and head init on every rank (DDP broadcasts too)
        dtype = comm.amp if comm.device.type == "cuda" else torch.float32
        trunk = load_trunk(C["base_dir"], C["keep"], dtype, comm.device)
        model = LoRAClassifier(trunk, C["layers"], n_classes, C["r"], C["alpha"], C["dropout"])
        on_gpu = comm.device.type == "cuda"
        return model, DDP(model, device_ids=[comm.device.index] if on_gpu else None)
    # XLA: build on the CPU (CPU RNG, so the same init everywhere), a few ranks at a time
    wave = C.get("load_parallel", 4)
    model = None
    for first in range(0, comm.world, wave):
        if first <= comm.rank < first + wave:
            torch.manual_seed(C["seed"])
            trunk = load_trunk(
                C["base_dir"], C["keep"], comm.amp, "cpu", attn_implementation="eager"
            )
            model = LoRAClassifier(
                trunk,
                C["layers"],
                n_classes,
                C["r"],
                C["alpha"],
                C["dropout"],
                ckpt=C.get("ckpt", False),
            ).to(comm.device)
            comm.sync()
            gc.collect()
        comm.barrier(f"load{first}")
    return model, model


def main(cfg_path: str):
    with open(cfg_path) as f:
        C = json.load(f)
    if C.get("accelerator") == "tpu":
        import torch_xla.distributed.xla_multiprocessing as xmp

        xmp.spawn(run, args=(C,))  # one process per TPU chip
    else:
        run(None, C)


def run(index, C: dict):
    global RANK
    comm = Comm(xla=C.get("accelerator") == "tpu")
    RANK = comm.rank
    rank, world, device, amp = comm.rank, comm.world, comm.device, comm.amp
    use_cuda = device.type == "cuda"
    T0 = time.time()
    log0(f"{world} processes on {device.type}, dtype {amp}, mode {C['mode']}")

    train = pd.read_csv(f"{C['comp']}/train.csv")
    pool = pd.read_parquet(C["pool"], columns=["text", "model", "match_weight"])
    pool_sample = sample_pool(pool, C["n_per_label"], C["pool_seed"])
    del pool
    if C.get("smoke"):
        train = train.groupby("model").head(12).reset_index(drop=True)
    labels = sorted(train.model.unique())
    l2i = {m: i for i, m in enumerate(labels)}
    assert len(labels) == 18 and set(pool_sample.model) <= set(labels)
    y_train = train.model.map(l2i).to_numpy()
    y_pool = pool_sample.model.map(l2i).to_numpy()
    folds = list(GroupKFold(n_splits=C["folds"]).split(train, y_train, train.prompt_id))
    log0(f"train {len(train)} in {C['folds']} folds, pool sample {len(pool_sample)}")

    tok = load_tokenizer(C["base_dir"])
    model, fwd = build_model(comm, C, len(labels))
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log0(
        f"{model.n_lora} LoRA layers, {n_train / 1e6:.1f}M trainable, loaded in "
        f"{time.time() - T0:.0f}s"
    )
    common = {"bs": C["bs"], "max_len": C["max_len"], "n_classes": len(labels), "amp_dtype": amp}
    out = C["out"]

    if use_cuda:
        torch.cuda.reset_peak_memory_stats()
    timing = C["mode"] == "timing"
    s1 = train_stage_ddp(
        comm,
        fwd,
        model,
        tok,
        pool_sample.text,
        y_pool,
        **C["stage1"],
        **common,
        seed=C["seed"],
        max_steps=C["timing_steps"] if timing else None,
        log_every=20 if timing else 200,
    )
    log0("stage 1:", s1)
    if use_cuda:
        log0(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
    if comm.xla and rank == 0:
        try:
            log0(f"TPU memory: {comm.xm.get_memory_info(device)}")
        except Exception as e:  # noqa: BLE001  (API differs between torch_xla versions)
            log0(f"TPU memory: not available ({e})")
    if timing:
        per_step = s1.get("steady_s_per_step", s1["seconds"] / s1["steps"])
        log0(f"steady {per_step:.2f}s/step (after the first steps and compiles)")
        bs = C["bs"]
        steps = {"stage 1": -(-len(pool_sample) // bs)}
        steps["stage 2, 5 folds"] = sum(2 * -(-len(f) // bs) for f, _ in folds)
        steps["stage 2 on all"] = 2 * -(-len(train) // bs)
        for k, v in steps.items():
            log0(f"{k}: {v} steps, ~{v * per_step / 60:.0f} min")
        log0(
            f"forecast for mode full: ~{sum(steps.values()) * per_step / 3600:.1f} h of "
            "training plus loading and scoring"
        )
        # then a short pass through everything mode full does, so a bug there shows up now
        log0("timing mode: a few stage-2 steps per fold to check the rest of the run")

    stage1_state = {k: v.clone() for k, v in model.trainable_state().items()}
    meta = {"labels": labels, "cfg": C, "world_size": world, "stage1": s1, "folds": []}
    if rank == 0:
        save_checkpoint(model, f"{out}/stage1", {"labels": labels, "cfg": C, "stage1": s1})

    s2_steps = 10 if timing else None
    oof = np.zeros((len(train), len(labels)), np.float32)
    fold_of = np.full(len(train), -1)
    for k, (fit_idx, hold_idx) in enumerate(folds):
        if timing:
            hold_idx = hold_idx[:48]
        model.load_state_dict({n: v.to(device) for n, v in stage1_state.items()}, strict=False)
        s2 = train_stage_ddp(
            comm,
            fwd,
            model,
            tok,
            train.text.iloc[fit_idx],
            y_train[fit_idx],
            **C["stage2"],
            **common,
            seed=C["seed"] + 1 + k,
            max_steps=s2_steps,
            log_every=100,
        )
        if rank == 0:
            z = predict_logits(
                model,
                tok,
                train.text.iloc[hold_idx],
                bs=2 * C["bs"],
                max_len=C["max_len"],
                amp_dtype=amp,
                static=comm.xla,
            )
            oof[hold_idx], fold_of[hold_idx] = z, k
            bacc = balanced_accuracy_score(y_train[hold_idx], z.argmax(1))
            meta["folds"].append(dict(fold=k, balanced_accuracy=bacc, n=len(hold_idx), **s2))
            save_checkpoint(
                model,
                f"{out}/fold{k}",
                {"labels": labels, "cfg": C, "fold": k, "holdout_balanced_accuracy": bacc},
            )
            log0(
                f"fold {k}: balanced accuracy {bacc:.4f} ({len(hold_idx)} passages) "
                f"after {(time.time() - T0) / 60:.0f} min"
            )
        comm.barrier(f"fold{k}")

    model.load_state_dict({n: v.to(device) for n, v in stage1_state.items()}, strict=False)
    s2_all = train_stage_ddp(
        comm,
        fwd,
        model,
        tok,
        train.text,
        y_train,
        **C["stage2"],
        **common,
        seed=C["seed"] + 1 + C["folds"],
        max_steps=s2_steps,
        log_every=100,
    )
    if rank == 0:
        done = fold_of >= 0  # every passage in mode full
        oof_bacc = balanced_accuracy_score(y_train[done], oof[done].argmax(1))
        meta.update(
            stage2_all=s2_all,
            oof_balanced_accuracy=oof_bacc,
            holdout_balanced_accuracy=meta["folds"][0]["balanced_accuracy"],
        )
        # the full fit sits at the top of the output folder, like the single-GPU runs
        save_checkpoint(model, out, meta)
        frame = pd.DataFrame(oof, columns=labels)
        frame.insert(0, "id", train.id.to_numpy())
        frame.insert(1, "model", train.model.to_numpy())
        frame.insert(2, "fold", fold_of)
        frame.to_parquet(f"{out}/oof_logits.parquet", index=False)
        frame[frame.fold == 0].drop(columns="fold").to_parquet(
            f"{out}/holdout_logits.parquet", index=False
        )
        log0(
            f"out-of-fold balanced accuracy {oof_bacc:.4f} over {done.sum()} passages; "
            f"per fold {[round(f['balanced_accuracy'], 4) for f in meta['folds']]}"
        )
        log0(f"saved to {out} in {(time.time() - T0) / 60:.1f} min")
    comm.barrier("done")
    comm.close()


if __name__ == "__main__":
    main(sys.argv[1])


## Train: one process per GPU

In `timing` mode stage 1 runs only 60 steps and prints a forecast for a full run; then every later step runs briefly (10 stage-2 steps per fold, 48 held-out passages scored per fold), so a bug there shows up in minutes, not hours.

In [ ]:
import subprocess
import sys
import threading

n_proc = torch.cuda.device_count() or 2  # 2 CPU processes for the local smoke test
env = dict(os.environ, MASTER_ADDR='127.0.0.1', MASTER_PORT='29517', WORLD_SIZE=str(n_proc),
           USE_LIBUV='0', PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false')
procs, logs = [], [open(f'rank{r}.log', 'w') for r in range(1, n_proc)]
for r in range(n_proc):
    out = subprocess.PIPE if r == 0 else logs[r - 1]
    procs.append(subprocess.Popen([sys.executable, 'lora_ddp.py', 'cfg.json'],
                                  env=dict(env, RANK=str(r), LOCAL_RANK=str(r)),
                                  stdout=out, stderr=subprocess.STDOUT, text=True))


def stream(p):
    for line in p.stdout:
        if 'UNEXPECTED' not in line:  # skip the per-layer load report of the cut-down model
            print(line, end='', flush=True)


reader = threading.Thread(target=stream, args=(procs[0],), daemon=True)
reader.start()
while any(p.poll() is None for p in procs):
    if any(p.poll() not in (None, 0) for p in procs):
        for p in procs:
            if p.poll() is None:
                p.kill()
        break
    time.sleep(5)
reader.join(timeout=30)
codes = [p.wait() for p in procs]
for f in logs:
    f.close()
for r in range(1, n_proc):
    if codes[r] != 0:
        print(f'--- rank {r} log ---')
        with open(f'rank{r}.log') as f:
            print(f.read()[-5000:])
assert codes == [0] * n_proc, f'training processes exited with {codes}'


## Summary

In [ ]:
for f in ['lora.py', 'lora_ddp.py', 'cfg.json'] + glob.glob('rank*.log'):
    os.remove(f)  # keep the output folder to the checkpoints
if MODE == 'timing':
    print('timing run: these scores come from 10 stage-2 steps per fold and only '
          'show that every step runs; set MODE to full for the real run')
meta = json.load(open(f"{CFG['out']}/meta.json"))
for fold in meta['folds']:
    print(f"fold {fold['fold']}: {fold['balanced_accuracy']:.4f} ({fold['n']} passages)")
print(f"out-of-fold: {meta['oof_balanced_accuracy']:.4f}")
print('outputs:', sorted(os.listdir(CFG['out'])))
print(f'done in {(time.time() - T0) / 60:.1f} min')